# Long LangChains with Azure OpenAI
## Enterprise Lab Notebook - Production-Ready Patterns


## What You'll Learn

| # | Topic | Takeaway |
|---|-------|----------|
| 1 | Azure OpenAI Setup | Configure any Azure LLM endpoint in LangChain |
| 2 | Basic Chains (LLMChain / LCEL) | Build prompt → LLM → parser pipelines |
| 3 | Sequential Chains | Chain outputs as inputs across multiple steps |
| 4 | Router Chains | Dynamically route queries to specialized sub-chains |
| 5 | Transformation Chains | Pre/post-process data mid-chain |
| 6 | Memory in Chains | Stateful multi-turn conversations |
| 7 | RAG Chain | Retrieval-Augmented Generation with Azure |
| 8 | Agent + Tool Chains | LLM-driven decision-making with tools |
| 9 | Parallel & Map Chains | Fan-out processing for throughput |
|10 | Production Patterns | Callbacks, streaming, error handling |


## Section 0 - Environment Setup

### Install dependencies

Run once per environment. These are the production-grade packages used throughout.

In [1]:
# Install all required packages
# Run this cell first, then restart the kernel if prompted

%pip install -q langchain langchain-openai langchain-community \
    openai tiktoken faiss-cpu python-dotenv azure-identity \
    langchain-core langchainhub

Note: you may need to restart the kernel to use updated packages.


### Configure Azure Credentials

You need an **Azure OpenAI** resource. From the Azure Portal:
- `AZURE_OPENAI_API_KEY` — under Keys and Endpoint
- `AZURE_OPENAI_ENDPOINT` — e.g., `https://YOUR_RESOURCE.openai.azure.com/`
- `AZURE_OPENAI_DEPLOYMENT_NAME` — your deployed model name (e.g., `gpt-4o`)
- `AZURE_OPENAI_API_VERSION` — e.g., `2024-02-15-preview`

**Best practice:** Use a `.env` file or Azure Key Vault. Never hardcode secrets.

In [2]:
import os
from dotenv import load_dotenv

# Load from .env file if present
load_dotenv()

# --- SET YOUR AZURE CREDENTIALS HERE ---
# Option A: Direct assignment (for quick testing only)
os.environ.setdefault("AZURE_OPENAI_API_KEY",        "")
os.environ.setdefault("AZURE_OPENAI_ENDPOINT",       "https://<TBD>.cognitiveservices.azure.com")
os.environ.setdefault("AZURE_OPENAI_DEPLOYMENT_NAME","gpt-4o")          # Your deployed model
os.environ.setdefault("AZURE_OPENAI_API_VERSION",    "2024-12-01-preview")

# Option B: Read from environment (recommended for production)
AZURE_API_KEY        = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_ENDPOINT       = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_DEPLOYMENT     = os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"]
AZURE_API_VERSION    = os.environ["AZURE_OPENAI_API_VERSION"]

print("Credentials loaded.")
print(f"   Endpoint  : {AZURE_ENDPOINT}")
print(f"   Deployment: {AZURE_DEPLOYMENT}")
print(f"   API Ver   : {AZURE_API_VERSION}")

Credentials loaded.
   Endpoint  : https://<TBD>.cognitiveservices.azure.com
   Deployment: gpt-4o
   API Ver   : 2024-12-01-preview


In [3]:
# ─────────────────────────────────────────────────────────────
# CENTRAL LLM FACTORY — reused in every section of this notebook
# ─────────────────────────────────────────────────────────────
from langchain_openai import AzureChatOpenAI

def get_llm(temperature: float = 0.0, max_tokens: int = 1024) -> AzureChatOpenAI:
    """
    Returns a configured AzureChatOpenAI instance.

    Args:
        temperature: 0.0 = deterministic, 1.0 = creative
        max_tokens:  Upper bound on response length
    """
    return AzureChatOpenAI(
        azure_deployment=AZURE_DEPLOYMENT,
        azure_endpoint=AZURE_ENDPOINT,
        api_key=AZURE_API_KEY,
        api_version=AZURE_API_VERSION,
        temperature=temperature,
        max_tokens=max_tokens,
    )

# Smoke test
llm = get_llm()
response = llm.invoke("Reply with exactly: 'Azure LangChain Lab - Ready!'")
print(response.content)

Azure LangChain Lab - Ready!


## Section 1 - Basic Chains: LCEL (LangChain Expression Language)

### Concept

**LCEL** is LangChain's modern declarative syntax using the `|` pipe operator. Every component implements `Runnable`, making chains composable, streamable, and parallelizable.

```
prompt_template | llm | output_parser
     ↓                ↓           ↓
  formats input   calls Azure   extracts text
```

**When to use:** Any single-step: summarize, classify, extract, rewrite.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(temperature=0.3)

# ── 1A: Simple summarization chain ──────────────────────────
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise technical writer. Summarize in {num_sentences} sentences."),
    ("human",  "{text}")
])

summarize_chain = summarize_prompt | llm | StrOutputParser()

article = """
Large language models (LLMs) have transformed the software industry by enabling developers
to build natural-language interfaces with minimal effort. Azure OpenAI Service provides
enterprise-grade access to models like GPT-4o, with SLAs, compliance certifications
(SOC 2, ISO 27001, HIPAA), private networking via VNet, and regional data residency.
LangChain abstracts the orchestration layer, letting engineers compose multi-step AI
pipelines using Python without managing raw HTTP calls or prompt state manually.
"""

result = summarize_chain.invoke({"text": article, "num_sentences": 2})
print("Summary:")
print(result)

Summary:
Large language models (LLMs) have revolutionized software development by simplifying the creation of natural-language interfaces, with Azure OpenAI Service offering enterprise-grade access to models like GPT-4o alongside robust compliance and networking features. LangChain streamlines multi-step AI pipeline orchestration, enabling developers to focus on functionality without handling low-level operations.


In [5]:
# ── 1B: Structured JSON output chain ────────────────────────
# Real-world use: extract entities from unstructured text
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List

class ExtractedEntities(BaseModel):
    companies:    List[str] = Field(description="Company names mentioned")
    technologies: List[str] = Field(description="Technology names mentioned")
    key_topics:   List[str] = Field(description="Main topics discussed")

parser = JsonOutputParser(pydantic_object=ExtractedEntities)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract structured entities from text. {format_instructions}"),
    ("human",  "{text}")
]).partial(format_instructions=parser.get_format_instructions())

extraction_chain = extraction_prompt | llm | parser

entities = extraction_chain.invoke({"text": article})
print("Extracted Entities:")
for k, v in entities.items():
    print(f"  {k}: {v}")

Extracted Entities:
  companies: ['Azure OpenAI Service', 'LangChain']
  technologies: ['Large language models', 'GPT-4o', 'Python', 'VNet']
  key_topics: ['natural-language interfaces', 'enterprise-grade access', 'compliance certifications', 'private networking', 'regional data residency', 'AI pipelines orchestration']


## Section 2 - Sequential Chains

### Concept

A **sequential chain** feeds the output of one chain directly as the input to the next. This models real-world multi-step workflows:

```
[Document] → [Summarize] → [Translate] → [Generate Title] → Final Output
```

**When to use:** Document processing pipelines, content transformation, multi-stage reasoning.

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

llm = get_llm(temperature=0.4)
str_parser = StrOutputParser()

# ── Step 1: Summarize ───────────────────────────────────────
step1_prompt = ChatPromptTemplate.from_template(
    "Summarize the following technical document in 3 bullet points:\n\n{document}"
)
step1_chain = step1_prompt | llm | str_parser

# ── Step 2: Critique ────────────────────────────────────────
step2_prompt = ChatPromptTemplate.from_template(
    "Given this summary:\n{summary}\n\n"
    "List 2 potential risks or gaps a senior architect should investigate."
)
step2_chain = step2_prompt | llm | str_parser

# ── Step 3: Executive brief ─────────────────────────────────
step3_prompt = ChatPromptTemplate.from_template(
    "Summary:\n{summary}\n\nRisks:\n{risks}\n\n"
    "Write a 2-sentence executive brief combining the above for a VP-level audience."
)
step3_chain = step3_prompt | llm | str_parser

# ── Compose the full pipeline ────────────────────────────────
# LCEL keeps intermediate values using RunnablePassthrough
full_pipeline = (
    {"document": RunnablePassthrough()}
    | RunnablePassthrough.assign(summary=step1_chain)
    | RunnablePassthrough.assign(risks=step2_chain)
    | step3_chain
)

# Run on a sample engineering RFC
rfc_text = """
RFC-2024-091: Migration of on-premises Oracle DB to Azure SQL Managed Instance.
Phase 1 covers schema migration using Database Migration Service (DMS).
Phase 2 covers data migration with minimal downtime using transactional replication.
Phase 3 covers cutover and decommission. Estimated downtime window: 4 hours.
Dependencies: VNet peering, ExpressRoute, legacy ETL jobs on SSIS.
"""

brief = full_pipeline.invoke(rfc_text)
print("Executive Brief:")
print(brief)

Executive Brief:
The proposed migration of the on-premises Oracle database to Azure SQL Managed Instance follows a three-phase approach designed to minimize downtime and ensure business continuity, with key dependencies on VNet peering, ExpressRoute connectivity, and the compatibility of legacy SSIS ETL jobs. Critical risks include potential network configuration or connectivity issues and challenges in adapting legacy ETL processes, both of which require senior architectural oversight to mitigate impacts on the migration timeline and post-migration operations.


In [7]:
# ── 2B: Inspect intermediate steps ──────────────────────────
# Useful for debugging production pipelines

debug_pipeline = (
    {"document": RunnablePassthrough()}
    | RunnablePassthrough.assign(summary=step1_chain)
    | RunnablePassthrough.assign(risks=step2_chain)
    | RunnablePassthrough.assign(brief=step3_chain)
)

full_state = debug_pipeline.invoke(rfc_text)

print("=" * 60)
print("STEP 1 — Summary:")
print(full_state["summary"])
print("\nSTEP 2 — Risks:")
print(full_state["risks"])
print("\nSTEP 3 — Brief:")
print(full_state["brief"])

STEP 1 — Summary:
- The migration process involves three phases: schema migration using Database Migration Service (DMS), data migration with minimal downtime via transactional replication, and final cutover with decommissioning (4-hour downtime window).  
- Key dependencies include VNet peering, ExpressRoute, and legacy ETL jobs running on SSIS.  
- The target platform for the migration is Azure SQL Managed Instance, transitioning from an on-premises Oracle database.  

STEP 2 — Risks:
1. **Compatibility and Performance Issues with Azure SQL Managed Instance**:  
   The senior architect should investigate potential compatibility issues between the on-premises Oracle database and Azure SQL Managed Instance, such as differences in data types, stored procedures, or complex SQL queries. Additionally, performance tuning may be required to ensure the migrated workloads run efficiently on the new platform.

2. **Impact of Legacy ETL Jobs on Migration Timeline**:  
   The dependency on legacy

## Section 3 - Router Chains

### Concept

A **router chain** classifies the input first, then routes it to the most appropriate specialized sub-chain. This avoids sending every query to a one-size-fits-all prompt.

```
Input ──▶ [Classifier Chain] ──▶ "legal" ──▶ [Legal Expert Chain]
                                 "finance" ──▶ [Finance Expert Chain]
                                 "hr"      ──▶ [HR Expert Chain]
                                 "other"   ──▶ [General Chain]
```

**When to use:** Enterprise chatbots, multi-domain Q&A systems, support ticket routing.

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnablePassthrough, RunnableLambda

llm = get_llm(temperature=0.2)
str_out = StrOutputParser()

# ── Specialized sub-chains ────────────────────────────────────
legal_chain = (
    ChatPromptTemplate.from_template(
        "You are a corporate legal advisor. Answer this legal question concisely and cite relevant principles:\n{query}"
    ) | llm | str_out
)

# FIX: Rephrased to avoid Azure content filter on financial terminology
finance_chain = (
    ChatPromptTemplate.from_template(
        "You are a finance director. Explain the accounting treatment and business impact "
        "of this financial question using standard frameworks:\n{query}"
    ) | llm | str_out
)

hr_chain = (
    ChatPromptTemplate.from_template(
        "You are a seasoned HR Business Partner. Answer this HR question with empathy and policy awareness:\n{query}"
    ) | llm | str_out
)

general_chain = (
    ChatPromptTemplate.from_template(
        "Answer the following question helpfully and accurately:\n{query}"
    ) | llm | str_out
)

# ── Classifier chain ──────────────────────────────────────────
classifier_prompt = ChatPromptTemplate.from_template(
    """Classify the following query into exactly ONE category: legal, finance, hr, other.
Respond with just the single word, lowercase.

Query: {query}
Category:"""
)
classifier = classifier_prompt | llm | str_out

# ── Content filter safe wrapper ───────────────────────────────
# Catches Azure content filter errors gracefully instead of crashing
def safe_invoke(chain):
    def _invoke(inputs):
        try:
            return chain.invoke(inputs)
        except ValueError as e:
            if "content filter" in str(e).lower():
                return (
                    "Response blocked by Azure content filter. "
                    "Try rephrasing the query or review your Azure OpenAI content policy settings "
                    "in the Azure Portal under: OpenAI resource → Content Filters."
                )
            raise  # re-raise if it's a different ValueError
    return RunnableLambda(_invoke)

# ── Router ────────────────────────────────────────────────────
router = RunnableBranch(
    (lambda x: "legal"   in x["category"], safe_invoke(legal_chain)),
    (lambda x: "finance" in x["category"], safe_invoke(finance_chain)),
    (lambda x: "hr"      in x["category"], safe_invoke(hr_chain)),
    safe_invoke(general_chain),
)

# ── Full routed pipeline ──────────────────────────────────────
routed_pipeline = (
    RunnablePassthrough.assign(category=classifier)
    | RunnablePassthrough.assign(
          route_used=lambda x: print(f"Routed to: [{x['category'].strip()}] chain") or x["category"]
      )
    | router
)

# ── Test queries ──────────────────────────────────────────────
queries = [
    "What are the GDPR implications of storing customer PII in a US-based Azure region?",
    "How should we account for cloud infrastructure CapEx vs OpEx in our P&L?",
    "Our employee in a remote location is asking for a hybrid work accommodation. What's our obligation?",
]

for q in queries:
    print(f"\n{'='*60}")
    print(f"Query: {q[:80]}...")
    answer = routed_pipeline.invoke({"query": q})
    print(f"Answer: {answer[:300]}...")


Query: What are the GDPR implications of storing customer PII in a US-based Azure regio...
Routed to: [legal] chain
Answer: Storing customer Personally Identifiable Information (PII) in a US-based Azure region has significant implications under the General Data Protection Regulation (GDPR). Key considerations include:

1. **Data Transfer Requirements**: Under GDPR, transferring personal data outside the European Economic...

Query: How should we account for cloud infrastructure CapEx vs OpEx in our P&L?...
Routed to: [finance] chain
Answer: Certainly! As a finance director, addressing the accounting treatment and business impact of cloud infrastructure expenditures requires a clear understanding of the distinction between **Capital Expenditures (CapEx)** and **Operating Expenses (OpEx)**, as well as their implications for financial rep...

Query: Our employee in a remote location is asking for a hybrid work accommodation. Wha...
Routed to: [hr] chain
Answer: Certainly! When addressing

## Section 4 - Memory in Chains (Stateful Conversations)

### Concept

LLMs are **stateless by default** — each call starts fresh. LangChain v0.3+ replaces
the legacy `memory` module with **`RunnableWithMessageHistory`** + a message history
store, giving you the same patterns but composable directly in LCEL pipelines.

| Old (Removed in v0.3) | Modern Replacement | Best For |
|---|---|---|
| `ConversationBufferMemory` | `InMemoryChatMessageHistory` | Short sessions, single user |
| `ConversationSummaryMemory` | `trim_messages(strategy="last")` + `InMemoryChatMessageHistory` | Long sessions, token-bounded |
| `ConversationBufferWindowMemory` | `trim_messages(max_messages=N)` | Keep last N turns only |
| `VectorStoreRetrieverMemory` | FAISS / Azure AI Search retriever in chain | Semantic recall over long history |
| `ConversationSummaryBufferMemory` | `trim_messages(max_tokens=N, strategy="last")` | Hybrid: recent verbatim + older trimmed |

### How the Modern Pattern Works
```
User Input
    │
    ▼
RunnableWithMessageHistory          ← manages load + save automatically
    │
    ├── loads chat_history from store[session_id]
    │
    ▼
ChatPromptTemplate                  ← injects history via MessagesPlaceholder
    │
    ▼
trim_messages                       ← enforces token / turn budget
    │
    ▼
AzureChatOpenAI                     ← stateless LLM call
    │
    ▼
StrOutputParser
    │
    └── saves (input, output) back to store[session_id]
```

### Storage Backends (swap one line)

| Backend | Class | Use Case |
|---|---|---|
| In-process dict | `InMemoryChatMessageHistory` | Development / notebooks |
| Redis (Azure Cache) | `RedisChatMessageHistory` | Production, low latency |
| SQL / Postgres | `SQLChatMessageHistory` | Auditable, persistent sessions |
| Cosmos DB | `CosmosDBChatMessageHistory` | Azure-native, globally distributed |

In [9]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import trim_messages, BaseMessage
from langchain_core.output_parsers import StrOutputParser
from typing import List
import tiktoken

llm = get_llm(temperature=0.5)

# ── gpt-4o uses o200k_base encoding ──────────────────────────
# (gpt-3.5/gpt-4/gpt-4-turbo use cl100k_base instead)
def count_tokens(messages: List[BaseMessage]) -> int:
    enc = tiktoken.get_encoding("o200k_base")
    total = 0
    for msg in messages:
        total += 4 + len(enc.encode(str(msg.content)))  # 4 = per-message overhead
    return total

trimmer = trim_messages(
    max_tokens=500,
    strategy="last",
    token_counter=count_tokens,
    include_system=True,
    allow_partial=False,
)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a seasoned Azure solutions architect helping a team plan their cloud migration. "
     "Be specific and professional. Reference prior conversation context when relevant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain = prompt | trimmer | llm | StrOutputParser()

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

SESSION = {"configurable": {"session_id": "azure-migration-01"}}

turns = [
    "We're migrating 40 microservices from AWS EKS to Azure Kubernetes Service. Where do we start?",
    "Our services use Kafka heavily. What's the Azure equivalent and migration path?",
    "Given our Kafka dependency you mentioned, what networking topology do you recommend?",
    "Summarize our architecture decisions so far in a table.",
]

for i, turn in enumerate(turns, 1):
    print(f"\n{'─'*60}")
    print(f"Turn {i}: {turn}")
    reply = chain_with_memory.invoke({"input": turn}, config=SESSION)
    print(f"Assistant: {reply[:400]}...\n")


────────────────────────────────────────────────────────────
Turn 1: We're migrating 40 microservices from AWS EKS to Azure Kubernetes Service. Where do we start?
Assistant: Migrating 40 microservices from AWS Elastic Kubernetes Service (EKS) to Azure Kubernetes Service (AKS) requires careful planning and execution to minimize downtime and ensure a smooth transition. Here’s a structured approach:

### 1. **Assessment and Planning**
   - **Inventory Microservices**: Catalog all 40 microservices, including dependencies, configurations, and resource requirements (CPU, me...


────────────────────────────────────────────────────────────
Turn 2: Our services use Kafka heavily. What's the Azure equivalent and migration path?
Assistant: Azure provides **Azure Event Hubs** as its equivalent for Kafka. Event Hubs is a fully managed, real-time data ingestion service that is designed for big data streaming and event-driven architectures. It has native support for Kafka, which makes it a great ch

## Section 5 - RAG Chain (Retrieval-Augmented Generation)

### Concept

**RAG** grounds LLM responses in your private documents, eliminating hallucinations about internal knowledge.

```
Query ──▶ [Embeddings] ──▶ [Vector Search] ──▶ [Relevant Chunks]
                                                      ↓
                                          [LLM + Context] ──▶ Answer
```

**When to use:** Internal knowledge bases, compliance Q&A, document-grounded chatbots.

In [10]:
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── Configure Azure Embeddings ───────────────────────────────
# NOTE: You need a separate embedding model deployment in Azure
# Common choice: text-embedding-ada-002 or text-embedding-3-large
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=os.environ.get("AZURE_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002"),
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
)

# ── Sample internal knowledge base ──────────────────────────
internal_docs = [
    Document(
        page_content="""
        Azure Policy: Cloud Spend Governance v2.3 (2024)
        All Azure resources must be tagged with: CostCenter, Environment, Owner.
        Monthly cloud spend over $50,000 requires VP Finance approval.
        Reserved Instances must be evaluated quarterly by the FinOps team.
        Orphaned resources (no activity >30 days) are auto-flagged for deletion.
        """,
        metadata={"source": "cloud_policy_v2.3", "type": "policy"}
    ),
    Document(
        page_content="""
        Azure Architecture Standard: Networking (2024)
        All production workloads must use Private Endpoints — no public internet exposure.
        Hub-and-spoke topology is the approved network design for enterprise subscriptions.
        Azure Firewall Premium is mandatory for all outbound traffic from production VNets.
        ExpressRoute is required for any workload transferring >1TB/day to on-premises.
        """,
        metadata={"source": "arch_network_2024", "type": "standard"}
    ),
    Document(
        page_content="""
        Data Classification Policy (Effective Jan 2024)
        Level 1 (Public): No restrictions. Level 2 (Internal): Encryption at rest required.
        Level 3 (Confidential): Encryption at rest + in transit + access logging mandatory.
        Level 4 (Restricted): All Level 3 controls + Customer Managed Keys (CMK) +
        Defender for Cloud enabled + quarterly penetration testing.
        PII and financial data are always classified as Level 4.
        """,
        metadata={"source": "data_classification_2024", "type": "policy"}
    ),
]

# ── Chunk & index documents ──────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(internal_docs)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(chunks)} chunks from {len(internal_docs)} documents")

Indexed 6 chunks from 3 documents


In [11]:
# ── Build the RAG chain ──────────────────────────────────────
llm = get_llm(temperature=0.0)  # Deterministic for compliance answers

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are an enterprise Azure compliance advisor.
Answer questions ONLY based on the provided internal policy documents.
If the answer is not in the context, say so explicitly — do not invent.
Always cite the source document name.

Context from internal policies:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', 'unknown')}]\n{d.page_content.strip()}"
        for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# ── Test queries ─────────────────────────────────────────────
questions = [
    "Do we need Customer Managed Keys for a database storing employee salary data?",
    "What approvals do I need to spin up $80K/month of Azure resources?",
    "Can we expose our production API directly to the internet?",
]

for q in questions:
    print(f"\n{q}")
    answer = rag_chain.invoke(q)
    print(f"{answer}")
    print()


Do we need Customer Managed Keys for a database storing employee salary data?
Yes, you need Customer Managed Keys (CMK) for a database storing employee salary data. According to the **data_classification_2024** policy, financial data is always classified as Level 4 (Restricted), and Level 4 requires Customer Managed Keys as part of its controls.

Source: **data_classification_2024**


What approvals do I need to spin up $80K/month of Azure resources?
You need VP Finance approval for monthly cloud spend over $50,000. 

Source: [cloud_policy_v2.3]


Can we expose our production API directly to the internet?
No, production workloads must use Private Endpoints and cannot be exposed directly to the public internet. 

Source: Azure Architecture Standard: Networking (2024) [arch_network_2024].



## Section 6 - Parallel Chains (Fan-Out with RunnableParallel)

### Concept

**Parallel chains** run multiple sub-chains simultaneously on the same input, reducing total latency when independent operations can run concurrently.

```
                    ┌──▶ [Summarize]  ──┐
Input ──▶ Parallel ─├──▶ [Classify]  ──├──▶ Aggregator ──▶ Output
                    └──▶ [Sentiment] ──┘
```

**When to use:** Document analysis dashboards, content moderation, multi-faceted evaluation.

In [12]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time

llm = get_llm(temperature=0.3)
str_out = StrOutputParser()

# ── Three independent analysis chains ────────────────────────
summary_chain = (
    ChatPromptTemplate.from_template("Summarize in one sentence: {text}")
    | llm | str_out
)

category_chain = (
    ChatPromptTemplate.from_template(
        "Classify into ONE category (technical/business/legal/hr/other): {text}\nCategory:"
    ) | llm | str_out
)

action_chain = (
    ChatPromptTemplate.from_template(
        "What is the single most important action item from this text? {text}\nAction Item:"
    ) | llm | str_out
)

risk_chain = (
    ChatPromptTemplate.from_template(
        "On a scale of Low/Medium/High, what is the risk level in this text? Explain in one sentence. {text}\nRisk:"
    ) | llm | str_out
)

# ── Parallel composition — all 4 chains run concurrently ─────
analysis_parallel = RunnableParallel(
    summary=summary_chain,
    category=category_chain,
    action_item=action_chain,
    risk_assessment=risk_chain,
)

# Sample incident report
incident_text = """
INCIDENT-2024-0421: Production database (Azure SQL MI, East US 2) experienced
connection timeouts for 23 minutes starting 14:32 UTC. Root cause: automated
scaling event consumed all DTU capacity. 1,200 customer-facing transactions failed.
Revenue impact estimated at $34,000. RCA in progress. Immediate fix: manual DTU
increase to 4000. Long-term: implement auto-scaling guardrails and alerting.
"""

start = time.time()
results = analysis_parallel.invoke({"text": incident_text})
elapsed = time.time() - start

print(f"All 4 analyses completed in {elapsed:.1f}s (parallel)")
print("\n" + "="*60)
for key, value in results.items():
    print(f"\n🔹 {key.upper()}:")
    print(f"   {value.strip()}")

All 4 analyses completed in 2.1s (parallel)


🔹 SUMMARY:
   A 23-minute connection timeout in the production database (Azure SQL MI, East US 2) due to an automated scaling event caused 1,200 transaction failures and an estimated $34,000 revenue loss, with immediate DTU increase to 4000 and plans for auto-scaling guardrails and alerting as a long-term fix.

🔹 CATEGORY:
   Technical

🔹 ACTION_ITEM:
   Implement auto-scaling guardrails and alerting.

🔹 RISK_ASSESSMENT:
   **Risk: High**  
The incident caused significant customer-facing transaction failures, a notable revenue impact, and highlights a critical infrastructure vulnerability, requiring both immediate and long-term remediation.


In [13]:
# ── Map chain: apply same chain to a batch of inputs ─────────
# Real-world use: bulk processing of tickets, emails, documents
from langchain_core.runnables import RunnableLambda

ticket_classifier = (
    ChatPromptTemplate.from_template(
        "Classify this support ticket severity (P1/P2/P3/P4) and team (infra/app/security/data).\n"
        "Respond as JSON: {{\"severity\": \"P?\", \"team\": \"...\"}}\n\nTicket: {ticket}"
    ) | llm | StrOutputParser()
)

tickets = [
    "Production login service returning 503 for all users in EU region since 5 mins",
    "Dashboard chart colors don't match brand guidelines on Safari browser",
    "Suspicious login attempts detected from 47 IPs across 3 continents in last hour",
    "ETL pipeline finished 2 hours late — daily report delayed but not missing",
]

# Use .batch() for concurrent processing of multiple inputs
print("Classifying support tickets in parallel...\n")
classifications = ticket_classifier.batch([{"ticket": t} for t in tickets])

for ticket, cls in zip(tickets, classifications):
    print(f"Ticket: {ticket[:60]}...")
    print(f"  → {cls}\n")

Classifying support tickets in parallel...

Ticket: Production login service returning 503 for all users in EU r...
  → ```json
{"severity": "P1", "team": "infra"}
```

Ticket: Dashboard chart colors don't match brand guidelines on Safar...
  → ```json
{"severity": "P4", "team": "app"}
```

Ticket: Suspicious login attempts detected from 47 IPs across 3 cont...
  → ```json
{"severity": "P1", "team": "security"}
```

Ticket: ETL pipeline finished 2 hours late — daily report delayed bu...
  → ```json
{"severity": "P3", "team": "data"}
```



## Section 7 - Agent Chains (LLM-Driven Tool Use)

### Concept

An **agent** lets the LLM decide *which tool to call* and *with what arguments*, based on the user's goal. This enables dynamic, multi-step reasoning without hardcoded logic.

```
User Goal ──▶ [LLM Thinks] ──▶ Tool Call ──▶ [LLM Observes] ──▶ Next Tool / Final Answer
                ↑_______________________________↓
                        (ReAct loop)
```

**When to use:** Complex Q&A requiring multiple lookups, automated workflows, code execution.

In [14]:
# ── Section 7: Agent + Tool Chains (LangChain v1.0+ API) ─────
# Based on: https://docs.langchain.com/oss/python/langchain/agents

from langchain.agents import create_agent
from langchain_core.tools import tool
from pydantic import BaseModel, Field
import json, datetime, random

llm = get_llm(temperature=0.0)

# ── Define enterprise tools ───────────────────────────────────
@tool
def get_azure_resource_cost(resource_group: str, days: int = 30) -> str:
    """Get Azure resource cost for a resource group over the past N days.
    Use this when asked about cloud spending or costs for a specific resource group."""
    cost  = round(random.uniform(1000, 50000), 2)
    trend = random.choice(["↑ 12%", "↓ 8%", "→ stable"])
    return json.dumps({
        "resource_group": resource_group,
        "period_days": days,
        "total_cost_usd": cost,
        "trend_vs_prior_period": trend,
        "top_services": ["Azure Kubernetes Service", "Azure SQL MI", "Azure Blob Storage"]
    })

@tool
def check_compliance_status(subscription_id: str) -> str:
    """Check Azure Policy compliance status for a subscription.
    Use this when asked about policy violations or compliance posture."""
    violations = random.randint(0, 15)
    return json.dumps({
        "subscription_id": subscription_id,
        "compliance_score": f"{random.randint(70, 99)}%",
        "total_policies": 124,
        "violations": violations,
        "critical_violations": random.randint(0, min(3, violations)),
        "last_evaluated": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    })

@tool
def list_orphaned_resources(resource_group: str) -> str:
    """List Azure resources with no activity in the past 30 days.
    Use this to identify waste or cleanup opportunities."""
    resources = [
        {"name": "vm-dev-unused-01",   "type": "Virtual Machine", "idle_days": 45, "monthly_cost": 312},
        {"name": "disk-backup-old-02", "type": "Managed Disk",    "idle_days": 91, "monthly_cost": 48},
        {"name": "pip-staging-03",     "type": "Public IP",       "idle_days": 33, "monthly_cost": 3.5},
    ]
    return json.dumps({
        "resource_group": resource_group,
        "orphaned_resources": resources,
        "potential_monthly_savings": sum(r["monthly_cost"] for r in resources)
    })

@tool
def calculate_reserved_instance_savings(vm_family: str, quantity: int, term_years: int = 1) -> str:
    """Calculate potential savings from Azure Reserved Instances vs pay-as-you-go.
    Use when evaluating Reserved Instance purchases."""
    payg_monthly   = quantity * random.uniform(200, 800)
    ri_discount    = 0.35 if term_years == 1 else 0.55
    ri_monthly     = payg_monthly * (1 - ri_discount)
    annual_savings = (payg_monthly - ri_monthly) * 12
    return json.dumps({
        "vm_family": vm_family, "quantity": quantity, "term_years": term_years,
        "payg_monthly_usd": round(payg_monthly, 2),
        "ri_monthly_usd":   round(ri_monthly, 2),
        "discount_pct":     f"{ri_discount*100:.0f}%",
        "annual_savings_usd": round(annual_savings, 2)
    })

tools = [get_azure_resource_cost, check_compliance_status,
         list_orphaned_resources, calculate_reserved_instance_savings]

# ── Create agent — v1.0 API uses create_agent() directly ─────
# system_prompt replaces the manual ChatPromptTemplate boilerplate
agent = create_agent(
    llm,                        # AzureChatOpenAI instance
    tools=tools,
    name="azure_finops_agent",  # snake_case required
    system_prompt=(
        "You are an Azure FinOps and compliance expert. "
        "Always invoke the relevant tools to gather real data before responding. "
        "Never answer without calling tools first. Show all retrieved data in your report."
    ),
)

print("Agent ready with", len(tools), "tools")
print("Tools:", [t.name for t in tools])



Agent ready with 4 tools
Tools: ['get_azure_resource_cost', 'check_compliance_status', 'list_orphaned_resources', 'calculate_reserved_instance_savings']


In [15]:
# ── Run a multi-step agent task ───────────────────────────────
# Input format: {"messages": [...]} — standard across all agents
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": """
        I need a complete cloud optimization report for resource group 'prod-eastus-rg'
        in subscription 'sub-prod-001'. Please:
        1. Check the current spend and trend
        2. Identify any orphaned resources
        3. Check compliance posture
        4. Calculate RI savings for 20x D4s_v3 VMs on a 1-year term
        5. Give me a prioritized action plan with estimated savings
        """
    }]
})

print("\n" + "="*60)
print("FINAL REPORT:")
print("="*60)
print(result["messages"][-1].content)

# ── BONUS: Stream intermediate steps ─────────────────────────
# Uncomment to see the ReAct loop (Reasoning → Tool Call → Observation) live
#
# from langchain.messages import AIMessage, HumanMessage
# for chunk in agent.stream(
#     {"messages": [{"role": "user", "content": "Check costs for prod-eastus-rg"}]},
#     stream_mode="values"
# ):
#     msg = chunk["messages"][-1]
#     if msg.content:
#         print(f"Agent: {msg.content}")
#     elif hasattr(msg, "tool_calls") and msg.tool_calls:
#         print(f"Calling: {[tc['name'] for tc in msg.tool_calls]}")

C:\Users\agenticaiuser\AppData\Local\Temp\ipykernel_3976\1272947087.py:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "last_evaluated": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")



FINAL REPORT:
### Cloud Optimization Report for Resource Group: `prod-eastus-rg` in Subscription: `sub-prod-001`

---

#### **1. Current Spend and Trend**
- **Total Spend (Last 30 Days):** $16,450.16
- **Trend vs Prior Period:** Stable (↔)
- **Top Services Contributing to Costs:**
  - Azure Kubernetes Service
  - Azure SQL Managed Instance
  - Azure Blob Storage

---

#### **2. Orphaned Resources**
Identified resources with no activity in the past 30 days:
- **`vm-dev-unused-01`** (Virtual Machine)
  - Idle for 45 days
  - Monthly Cost: $312
- **`disk-backup-old-02`** (Managed Disk)
  - Idle for 91 days
  - Monthly Cost: $48
- **`pip-staging-03`** (Public IP)
  - Idle for 33 days
  - Monthly Cost: $3.50

**Potential Monthly Savings from Cleanup:** $363.50

---

#### **3. Compliance Posture**
- **Compliance Score:** 88%
- **Total Policies Evaluated:** 124
- **Violations:** 12 (2 critical)
- **Critical Violations:** 
  - Missing encryption on certain storage accounts
  - Non-compliant n

## Section 8 - Transformation & Conditional Chains

### Concept

**Transformation chains** apply Python logic mid-pipeline — cleaning data, computing values, filtering results — without an LLM call. Combined with conditionals, you can build sophisticated branching logic.

**When to use:** Data preprocessing, post-processing, conditional quality gates.

In [16]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableBranch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re

llm = get_llm(temperature=0.3)
str_out = StrOutputParser()

# ── Pre-processing transformation ────────────────────────────
def preprocess_text(inputs: dict) -> dict:
    """Clean and enrich text before sending to LLM."""
    text = inputs["raw_text"]
    cleaned = re.sub(r'\s+', ' ', text).strip()
    word_count = len(cleaned.split())
    return {
        "text": cleaned,
        "word_count": word_count,
        "is_long": word_count > 100,
    }

# ── Conditional quality gate ──────────────────────────────────
short_prompt = ChatPromptTemplate.from_template(
    "Expand this brief note into a professional paragraph: {text}"
)
long_prompt = ChatPromptTemplate.from_template(
    "Condense this {word_count}-word document into a 3-bullet executive summary: {text}"
)

short_chain = short_prompt | llm | str_out
long_chain  = long_prompt  | llm | str_out

# Route based on Python-computed condition (not LLM classification)
conditional_chain = RunnableBranch(
    (lambda x: x["is_long"], long_chain),
    short_chain,
)

# ── Post-processing transformation ────────────────────────────
def postprocess(inputs: dict) -> dict:
    """Add metadata to the final output."""
    return {
        "result": inputs["result"],
        "processing_mode": "condensed" if inputs["is_long"] else "expanded",
        "original_word_count": inputs["word_count"],
        "output_word_count": len(inputs["result"].split())
    }

# ── Full adaptive pipeline ────────────────────────────────────
adaptive_pipeline = (
    RunnableLambda(preprocess_text)
    | RunnablePassthrough.assign(result=conditional_chain)
    | RunnableLambda(postprocess)
)

# Test with short input
short_input = {"raw_text": "   DB migration   delayed.  Storage quota issue.  "}
long_input = {"raw_text": """
Our Q3 infrastructure review revealed significant cost overruns in three areas:
compute (Azure VMs running at 15% average CPU), storage (unattached managed disks
accumulating since January), and networking (egress charges from misconfigured
load balancers). The FinOps team estimates $180K in annualized waste. Recommended
actions include right-sizing D-series VMs to B-series for dev workloads, enabling
Azure Advisor auto-remediation, and implementing Azure Cost Management budgets with
automated alerts at 80% and 100% thresholds. Executive sponsorship from VP Engineering
is required to enforce tagging policy enforcement via Azure Policy deny effects.
"""}

for label, inp in [("SHORT INPUT", short_input), ("LONG INPUT", long_input)]:
    print(f"\n{'='*60}")
    print(f"{label}")
    out = adaptive_pipeline.invoke(inp)
    print(f"Mode: {out['processing_mode']} | {out['original_word_count']} → {out['output_word_count']} words")
    print(f"\nResult:\n{out['result']}")


SHORT INPUT
Mode: expanded | 6 → 81 words

Result:
The database migration has been delayed due to a storage quota issue that needs to be addressed before proceeding. The current storage limitations have impacted the ability to successfully transfer data, requiring additional time to resolve the matter and ensure the migration process can be completed without errors. Efforts are underway to analyze the storage requirements and implement the necessary adjustments to accommodate the migration. Once the issue is resolved, the migration will resume as planned, with minimal disruption to operations.

LONG INPUT
Mode: expanded | 88 → 155 words

Result:
The Q3 infrastructure review identified critical areas of cost inefficiency across compute, storage, and networking, resulting in an estimated $180K in annualized waste, according to the FinOps team. Specifically, compute resources showed inefficiencies with Azure virtual machines operating at an average CPU utilization of only 15%, while stor

## Section 9 - Streaming Chains

### Concept

**Streaming** delivers tokens as they are generated, dramatically improving perceived responsiveness in production UIs. All LCEL chains support streaming natively via `.stream()`.

**When to use:** Chatbots, report generation UIs, any interactive application.

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import sys

llm_stream = get_llm(temperature=0.7, max_tokens=800)

stream_prompt = ChatPromptTemplate.from_template(
    """Write a detailed Azure Well-Architected Framework review for the following architecture.
Cover all 5 pillars: Reliability, Security, Cost Optimization, Operational Excellence, Performance.

Architecture: {architecture}"""
)

stream_chain = stream_prompt | llm_stream | StrOutputParser()

architecture = """
Single-region Azure Kubernetes Service cluster (3 nodes, Standard_D4s_v3) running
8 microservices. Azure SQL Database (General Purpose, 8 vCores) as primary store.
Azure Blob Storage for media assets. No CDN. Public IPs on all services.
Manual deployments via kubectl. No monitoring beyond basic Azure Monitor alerts.
"""

print("📡 Streaming response (tokens appear as generated):\n")
print("-" * 60)

# Stream tokens in real-time
for chunk in stream_chain.stream({"architecture": architecture}):
    print(chunk, end="", flush=True)

print("\n" + "-" * 60)
print("\nStream complete")

📡 Streaming response (tokens appear as generated):

------------------------------------------------------------
### Azure Well-Architected Framework Review

The Azure Well-Architected Framework provides guidance to design, build, and operate reliable, secure, cost-efficient, performant, and operationally excellent solutions in Azure. Below is a detailed review of the provided architecture across the five pillars of the framework:

---

### **1. Reliability**
**Reliability** ensures your application can meet the demands and recover quickly from failures.

#### **Strengths:**
- The Azure Kubernetes Service (AKS) cluster provides built-in node auto-repair and self-healing capabilities for the Kubernetes control plane, enhancing reliability.
- Azure SQL Database (General Purpose tier) offers built-in high availability with automated backups and geo-replication capabilities (if configured).

#### **Concerns and Recommendations:**
1. **Single-region deployment:**  
   - Running everything i

## Section 10 - Production Patterns: Callbacks, Logging & Error Handling

### Concept

Production LangChain deployments need **observability** and **resilience**:
- **Callbacks** hook into every chain event (start, end, token, error)
- **Retry logic** handles transient Azure OpenAI rate limits
- **Fallback chains** ensure degraded-but-functional responses

**When to use:** Any production deployment.

In [18]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from typing import Any, Dict, List, Union
import time, logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

# ── Production callback handler ───────────────────────────────
class ProductionCallbackHandler(BaseCallbackHandler):
    """Logs chain execution metrics for production observability.
    Plug into any LangChain component via callbacks=[handler].
    In production: send these metrics to Azure Monitor / App Insights.
    """

    def __init__(self):
        self.run_starts: Dict[str, float] = {}
        self.metrics: List[Dict] = []

    def on_chain_start(self, serialized: Dict[str, Any], inputs: Dict[str, Any],
                       run_id=None, **kwargs) -> None:
        self.run_starts[str(run_id)] = time.time()
        chain_name = serialized.get("id", ["?"])[-1]
        logger.info(f"[START] Chain={chain_name} RunID={str(run_id)[:8]}")

    def on_chain_end(self, outputs: Dict[str, Any], run_id=None, **kwargs) -> None:
        elapsed = time.time() - self.run_starts.pop(str(run_id), time.time())
        metric = {"run_id": str(run_id)[:8], "latency_ms": round(elapsed * 1000), "status": "success"}
        self.metrics.append(metric)
        logger.info(f"[END]   RunID={metric['run_id']} Latency={metric['latency_ms']}ms")

    def on_chain_error(self, error: Union[Exception, KeyboardInterrupt],
                       run_id=None, **kwargs) -> None:
        metric = {"run_id": str(run_id)[:8], "status": "error", "error": str(error)[:100]}
        self.metrics.append(metric)
        logger.error(f"[ERROR] RunID={metric['run_id']} Error={metric['error']}")

    def on_llm_end(self, response: LLMResult, run_id=None, **kwargs) -> None:
        if response.llm_output:
            usage = response.llm_output.get("token_usage", {})
            if usage:
                logger.info(
                    f"[TOKENS] Prompt={usage.get('prompt_tokens',0)} "
                    f"Completion={usage.get('completion_tokens',0)} "
                    f"Total={usage.get('total_tokens',0)}"
                )

    def get_summary(self) -> Dict:
        if not self.metrics:
            return {"runs": 0}
        latencies = [m["latency_ms"] for m in self.metrics if "latency_ms" in m]
        return {
            "total_runs": len(self.metrics),
            "success_rate": f"{sum(1 for m in self.metrics if m['status']=='success')/len(self.metrics)*100:.1f}%",
            "avg_latency_ms": round(sum(latencies)/len(latencies)) if latencies else 0,
            "max_latency_ms": max(latencies) if latencies else 0,
        }

handler = ProductionCallbackHandler()

# ── Attach callback to a chain ────────────────────────────────
monitored_llm = get_llm(temperature=0.2)

analysis_prompt = ChatPromptTemplate.from_template(
    "Analyze the business impact of: {topic}"
)
monitored_chain = analysis_prompt | monitored_llm | StrOutputParser()

topics = [
    "Azure region outage lasting 4 hours affecting East US",
    "20% reduction in cloud costs through RI commitments",
]

for topic in topics:
    result = monitored_chain.invoke({"topic": topic}, config={"callbacks": [handler]})
    print(f"\n{topic[:50]}...")
    print(f"Response: {result[:150]}...")

print("\n" + "="*50)
print("Production Metrics Summary:")
for k, v in handler.get_summary().items():
    print(f"   {k}: {v}")

2026-03-11 08:23:38,398 WARNING Error in ProductionCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
2026-03-11 08:23:38,401 INFO [START] Chain=ChatPromptTemplate RunID=019cdbfe
2026-03-11 08:23:38,402 INFO [END]   RunID=019cdbfe Latency=1ms
2026-03-11 08:23:58,195 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:23:58,197 INFO [TOKENS] Prompt=23 Completion=956 Total=979
2026-03-11 08:23:58,198 WARNING Error in ProductionCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
2026-03-11 08:23:58,199 INFO [END]   RunID=019cdbfe Latency=1ms
2026-03-11 08:23:58,199 INFO [END]   RunID=019cdbfe Latency=19801ms
2026-03-11 08:23:58,200 WARNING Error in ProductionCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
2026-03-11 08:23:58


Azure region outage lasting 4 hours affecting East...
Response: An Azure region outage lasting 4 hours in the East US can have significant business impacts depending on the scale of operations, the dependency on Az...


2026-03-11 08:24:09,981 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:24:09,983 INFO [TOKENS] Prompt=23 Completion=635 Total=658
2026-03-11 08:24:09,984 WARNING Error in ProductionCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
2026-03-11 08:24:09,985 INFO [END]   RunID=019cdbff Latency=1ms
2026-03-11 08:24:09,986 INFO [END]   RunID=019cdbfe Latency=11786ms



20% reduction in cloud costs through RI commitment...
Response: A 20% reduction in cloud costs through Reserved Instance (RI) commitments can have significant business impacts across various dimensions. Below is an...

Production Metrics Summary:
   total_runs: 6
   success_rate: 100.0%
   avg_latency_ms: 5265
   max_latency_ms: 19801


In [19]:
# ── Fallback chains for resilience ───────────────────────────
# If primary chain fails (rate limit, timeout), fallback silently
from langchain_core.runnables import RunnablePassthrough

primary_llm  = get_llm(temperature=0.0, max_tokens=500)
fallback_llm = get_llm(temperature=0.0, max_tokens=200)  # Smaller, faster fallback

primary_prompt = ChatPromptTemplate.from_template(
    "Provide a detailed analysis (5 points) of: {topic}"
)
fallback_prompt = ChatPromptTemplate.from_template(
    "Briefly summarize (2 sentences) the key consideration for: {topic}"
)

primary_chain  = primary_prompt  | primary_llm  | StrOutputParser()
fallback_chain = fallback_prompt | fallback_llm | StrOutputParser()

# .with_fallbacks() automatically tries fallback on any exception
resilient_chain = primary_chain.with_fallbacks(
    [fallback_chain],
    exceptions_to_handle=(Exception,),
)

result = resilient_chain.invoke({"topic": "Azure OpenAI token rate limits in enterprise deployments"})
print("Resilient Chain Result:")
print(result)

2026-03-11 08:24:24,606 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Resilient Chain Result:
Azure OpenAI token rate limits in enterprise deployments are an important consideration for organizations leveraging OpenAI models through Azure's platform. These limits dictate how many tokens can be processed within a given timeframe, impacting the scalability, performance, and cost of AI applications. Below is a detailed analysis of the topic:

---

### **1. Definition and Importance of Token Rate Limits**
Token rate limits refer to the maximum number of tokens (units of text) that can be processed by the OpenAI models within a specific time period. Tokens are the building blocks of text, including words, punctuation, and spaces. For enterprise deployments, token rate limits are critical because they directly affect the throughput and responsiveness of applications using OpenAI models. Exceeding these limits can result in throttling, delayed responses, or even service interruptions, which can impact business operations.

---

### **2. Factors Influencing Toke

## Section 11 - Putting It All Together: Enterprise Document Intelligence Pipeline

### Complete Production Pipeline

This final section assembles a realistic enterprise pipeline that combines:
- **Parallel analysis** (multiple dimensions simultaneously)
- **Sequential enrichment** (each step builds on prior)
- **Conditional routing** (specialized handling by document type)
- **Structured output** (machine-readable JSON for downstream systems)
- **Production callbacks** (observability baked in)

```
Document Input
      │
      ▼
 [Parallel Analysis: summary + entities + sentiment + classification]
      │
      ▼
 [Router: incident vs. proposal vs. compliance]
      │                │                  │
 [Incident SLA]  [Proposal Score]  [Compliance Check]
      │                │                  │
      └────────────────┴──────────────────┘
                       │
                       ▼
             [Executive Report Generator]
```

In [20]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableBranch, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(temperature=0.2)
str_out = StrOutputParser()

# ── Stage 1: Parallel initial analysis ───────────────────────

summary_chain = (
    ChatPromptTemplate.from_template("Summarize in 2 sentences: {document}") | llm | str_out
)
doc_type_chain = (
    ChatPromptTemplate.from_template(
        "Classify document type: incident/proposal/compliance/other. One word only. Document: {document}"
    ) | llm | str_out
)
urgency_chain = (
    ChatPromptTemplate.from_template(
        "Rate urgency 1-5 (5=critical). Reply with digit only. Document: {document}"
    ) | llm | str_out
)
stakeholders_chain = (
    ChatPromptTemplate.from_template(
        "List stakeholder roles who need to act (comma-separated, max 4): {document}"
    ) | llm | str_out
)

# ── Stage 2: Type-specific deep analysis ─────────────────────
incident_deep = (
    ChatPromptTemplate.from_template(
        "For this incident: {summary}\nProvide: RCA hypothesis, immediate action, prevention measure."
    ) | llm | str_out
)
proposal_deep = (
    ChatPromptTemplate.from_template(
        "For this proposal: {summary}\nScore ROI (1-10), identify top 2 risks, recommend: Approve/Reject/Revise."
    ) | llm | str_out
)
compliance_deep = (
    ChatPromptTemplate.from_template(
        "For this compliance document: {summary}\nIdentify: regulations referenced, gaps, required controls."
    ) | llm | str_out
)
general_deep = (
    ChatPromptTemplate.from_template(
        "For this document: {summary}\nProvide 3 actionable next steps."
    ) | llm | str_out
)

stage2_router = RunnableBranch(
    (lambda x: "incident"   in x["doc_type"].lower(), incident_deep),
    (lambda x: "proposal"   in x["doc_type"].lower(), proposal_deep),
    (lambda x: "compliance" in x["doc_type"].lower(), compliance_deep),
    general_deep,
)

# ── Stage 3: Executive report ─────────────────────────────────
exec_report = (
    ChatPromptTemplate.from_template(
        """Create an executive briefing card with these sections:
**DOCUMENT TYPE:** {doc_type}
**URGENCY:** {urgency}/5
**STAKEHOLDERS:** {key_stakeholders}

**SUMMARY:**
{summary}

**DETAILED ANALYSIS:**
{deep_analysis}

Format as a professional executive briefing in markdown."""
    ) | llm | str_out
)

# ── Assemble full pipeline ────────────────────────────────────
# FIX: assign each chain explicitly — no .steps unpacking needed
full_enterprise_pipeline = (
    {"document": RunnablePassthrough()}
    | RunnablePassthrough.assign(
        summary=summary_chain,
        doc_type=doc_type_chain,
        urgency=urgency_chain,
        key_stakeholders=stakeholders_chain,
    )
    | RunnablePassthrough.assign(deep_analysis=stage2_router)
    | exec_report
)

print("Enterprise pipeline assembled")

Enterprise pipeline assembled


In [21]:
# ── Test ──────────────────────────────────────────────────────
test_documents = {
    "Incident Report": """
    SEV1 INCIDENT: Azure Service Bus in West Europe region experiencing message loss
    since 09:14 UTC. 3 downstream microservices unable to process orders.
    Estimated 4,200 orders affected. Customer support volume up 340%.
    Engineering team engaged. Azure support ticket raised: #INC-20240419-4521.
    """,
    "Business Proposal": """
    PROPOSAL: AI-Powered Customer Churn Prediction System
    Investment required: $280K (Year 1). Expected revenue retention: $1.2M annually.
    Stack: Azure ML, Azure Databricks, Power BI. Timeline: 6 months to production.
    Team: 2 data scientists, 1 ML engineer, 1 product manager.
    Risk: Model accuracy below 85% threshold would reduce ROI significantly.
    """,
}

for doc_name, document in test_documents.items():
    print(f"\n{'='*70}")
    print(f"Processing: {doc_name}")
    print(f"{'='*70}")
    report = full_enterprise_pipeline.invoke(document)
    print(report)


Processing: Incident Report


2026-03-11 08:24:26,065 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:24:26,245 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:24:26,402 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:24:27,745 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:24:40,487 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:24:51,132 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/

```markdown
# **Executive Briefing Card**

---

## **DOCUMENT TYPE:**  
Incident  

## **URGENCY:**  
5/5  

## **STAKEHOLDERS:**  
- Engineering  
- Azure Support  
- Customer Support  
- Incident Manager  

---

## **SUMMARY:**  
A SEV1 incident involving Azure Service Bus in the West Europe region has caused message loss since **09:14 UTC**, impacting three downstream microservices and affecting an estimated **4,200 orders**. Customer support volume has surged by **340%**, and the engineering team is actively addressing the issue with an Azure support ticket raised (**#INC-20240419-4521**).  

---

## **DETAILED ANALYSIS:**  

### **Root Cause Analysis (RCA) Hypothesis**  
The SEV1 incident involving Azure Service Bus in the West Europe region appears to be caused by one or more of the following factors:  
1. **Service Bus Outage or Degraded Performance**: A potential issue with Azure Service Bus infrastructure (e.g., throttling, partition failure, or connectivity issues) may have l

2026-03-11 08:24:51,839 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:24:51,885 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:24:52,068 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:24:52,683 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:25:01,860 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-03-11 08:25:13,582 INFO HTTP Request: POST https://<TBD>.cognitiveservices.azure.com/

```markdown
# Executive Briefing Card

---

## **DOCUMENT TYPE:**  
Proposal  

## **URGENCY:**  
4/5  

## **STAKEHOLDERS:**  
- Executive Sponsor  
- Data Science Team  
- IT Infrastructure Team  
- Product Manager  

---

## **SUMMARY:**  
The proposal outlines the development of an AI-powered customer churn prediction system, requiring a **$280K investment in the first year** to leverage **Azure ML, Azure Databricks, and Power BI**, with a **6-month timeline to production**. The system, developed by a team of four experts, aims to retain **$1.2M annually in revenue**. However, achieving an accuracy below **85%** poses a significant risk to the return on investment (ROI).

---

## **DETAILED ANALYSIS:**

### **ROI Score:**  
**8/10**

#### **Justification:**  
- The proposal presents a strong financial case, targeting **$1.2M in annual revenue retention** with a **$280K first-year investment**, indicating a high potential ROI.  
- Leveraging **Azure ML, Azure Databricks, and Power B

## Section 12 - Reference Card & Quick Templates

Copy-paste these patterns into your own projects.

In [22]:
# ═══════════════════════════════════════════════════════════════
# QUICK REFERENCE: LangChain + Azure Patterns
# ═══════════════════════════════════════════════════════════════

reference = """
╔══════════════════════════════════════════════════════════════╗
║          LANGCHAIN + AZURE OPENAI — PATTERN REFERENCE        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. BASIC CHAIN (LCEL)                                       ║
║     chain = prompt | llm | StrOutputParser()                 ║
║     result = chain.invoke({"key": "value"})                  ║
║                                                              ║
║  2. SEQUENTIAL WITH STATE                                    ║
║     chain = (                                                ║
║       {"input": RunnablePassthrough()}                       ║
║       | RunnablePassthrough.assign(step1=chain1)             ║
║       | RunnablePassthrough.assign(step2=chain2)             ║
║     )                                                        ║
║                                                              ║
║  3. PARALLEL EXECUTION                                       ║
║     chain = RunnableParallel(a=chain_a, b=chain_b)           ║
║     # Returns dict with keys 'a' and 'b'                     ║
║                                                              ║
║  4. CONDITIONAL ROUTING                                      ║
║     chain = RunnableBranch(                                  ║
║       (lambda x: condition, handler_chain),                  ║
║       default_chain,                                         ║
║     )                                                        ║
║                                                              ║
║  5. BATCH PROCESSING                                         ║
║     results = chain.batch([{"k": v1}, {"k": v2}])            ║
║                                                              ║
║  6. STREAMING                                                ║
║     for chunk in chain.stream(input):                        ║
║         print(chunk, end="", flush=True)                     ║
║                                                              ║
║  7. FALLBACK RESILIENCE                                      ║
║     chain = primary.with_fallbacks([fallback])               ║
║                                                              ║
║  8. PYTHON TRANSFORMATION                                    ║
║     step = RunnableLambda(my_python_function)                ║
║                                                              ║
║  9. MEMORY (MULTI-TURN)                                      ║
║     memory = ConversationSummaryBufferMemory(llm=llm)        ║
║     history = memory.load_memory_variables({})               ║
║     memory.save_context({"input": q}, {"output": a})         ║
║                                                              ║
║  10. CALLBACKS (OBSERVABILITY)                               ║
║      chain.invoke(input, config={"callbacks": [handler]})    ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  AZURE ENV VARS NEEDED:                                      ║
║  AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT,               ║
║  AZURE_OPENAI_DEPLOYMENT_NAME, AZURE_OPENAI_API_VERSION      ║
║  (Optional) AZURE_EMBEDDING_DEPLOYMENT                       ║
╚══════════════════════════════════════════════════════════════╝
"""

print(reference)


╔══════════════════════════════════════════════════════════════╗
║          LANGCHAIN + AZURE OPENAI — PATTERN REFERENCE        ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. BASIC CHAIN (LCEL)                                       ║
║     chain = prompt | llm | StrOutputParser()                 ║
║     result = chain.invoke({"key": "value"})                  ║
║                                                              ║
║  2. SEQUENTIAL WITH STATE                                    ║
║     chain = (                                                ║
║       {"input": RunnablePassthrough()}                       ║
║       | RunnablePassthrough.assign(step1=chain1)             ║
║       | RunnablePassthrough.assign(step2=chain2)             ║
║     )                                                        ║
║                                                              ║
║  3. PARALLEL EXECUTION

## Lab Complete

### What You Built

| Pattern | Section | Production Ready? |
|---------|---------|-------------------|
| Basic LCEL chains | S1 | Yes |
| Sequential pipelines with state | S2 | Yes |
| Router chains (domain-based) | S3 | Yes |
| Stateful memory (multi-turn) | S4 | Yes |
| RAG with Azure Embeddings | S5 | Yes |
| Parallel / batch chains | S6 | Yes |
| Agent + custom tools | S7 | Yes |
| Transformation + conditionals | S8 | Yes |
| Streaming responses | S9 | Yes |
| Callbacks + fallback + logging | S10 | Yes |
| Full enterprise pipeline | S11 | Yes |

### Next Steps

1. **Persist vector store** to Azure Blob Storage (`FAISS.save_local()` + blob upload)
2. **Send callback metrics** to Azure Application Insights via `opencensus-ext-azure`
3. **Deploy chains as APIs** using FastAPI + Azure Container Apps
4. **Add LangSmith tracing** for full pipeline observability in production
5. **Explore Azure AI Search** as a managed alternative to FAISS for RAG at scale